# SpectraShift Week 7: freeze foundation contracts
Use CPU with Internet enabled. Attach source v6, Week 2 frozen data, Week 5 contracts, Week 6 contracts, and Week 6 complete.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, yaml

INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent / 'src/spectrashift/train/week7.py').is_file()]
if not projects:
    bundles = sorted(INPUT.rglob('spectrashift-kaggle-source.zip'))
    assert len(bundles) == 1, f'Expected one Week 7 source bundle, found {bundles}'
    source_work = Path('/tmp/spectrashift-week7-source')
    if source_work.exists(): shutil.rmtree(source_work)
    shutil.unpack_archive(str(bundles[0]), str(source_work))
    projects = [source_work]
assert projects, 'No Week 7 source tree found'
PROJECT = sorted(projects, key=lambda path: len(str(path)))[0]
sys.path.insert(0, str(PROJECT / 'src'))
os.chdir(PROJECT)

def unique_file(name):
    candidates = sorted(INPUT.rglob(name))
    by_hash = {}
    for path in candidates:
        by_hash.setdefault(hashlib.sha256(path.read_bytes()).hexdigest(), path)
    assert len(by_hash) == 1, f'Expected one unique {name}; found {candidates}'
    return next(iter(by_hash.values()))

def install_offline_foundation_dependencies():
    wheels = sorted(INPUT.rglob('foundation-wheels')) + sorted(Path('/kaggle/working').rglob('foundation-wheels'))
    if not wheels: return
    missing = []
    for module, package in [('upath','universal-pathlib'), ('omegaconf','omegaconf'), ('iopath','iopath'), ('fvcore','fvcore'), ('einops','einops'), ('huggingface_hub','huggingface_hub')]:
        try: __import__(module)
        except ImportError: missing.append(package)
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--find-links', str(wheels[0]), *missing])

from urllib.request import urlretrieve

WORK = Path('/kaggle/working/spectrashift-week7-contracts')
ASSETS = WORK / 'assets'
ASSETS.mkdir(parents=True, exist_ok=True)
MANIFEST = unique_file('partitions.parquet')
NORMALIZATION = unique_file('normalization.json')
FREEZE = unique_file('freeze_summary.json')
WEEK5_CONTRACTS = unique_file('week5_contracts_summary.json')
WEEK6_SUMMARY = unique_file('week6_run_summary.json')
RGB_CONTRACT = unique_file('rgb_percentile_contract.json')
STAGED = next(path.parent for path in INPUT.rglob('staging_summary.json'))
config = yaml.safe_load((PROJECT / 'configs/downstream/week7.yaml').read_text())
config['data'].update({'manifest_path': str(MANIFEST), 'staged_root': str(STAGED), 'normalization_path': str(NORMALIZATION), 'freeze_summary_path': str(FREEZE)})
config['contracts'].update({'output_dir': str(WORK), 'week5_contracts_summary_path': str(WEEK5_CONTRACTS), 'week6_summary_path': str(WEEK6_SUMMARY), 'rgb_contract_path': str(RGB_CONTRACT)})

downloads = {
    'dinov2-source.zip': config['assets']['dinov2_source_url'],
    'dinov2_vits14_pretrain.pth': config['assets']['dinov2_weights_url'],
    'olmoearth-minimal.zip': config['assets']['olmoearth_minimal_url'],
}
for name, url in downloads.items():
    target = ASSETS / name
    if not target.exists(): urlretrieve(url, target)
for archive, directory in [('dinov2-source.zip','dinov2-source'), ('olmoearth-minimal.zip','olmoearth-minimal-source')]:
    target = ASSETS / directory
    if not target.exists():
        target.mkdir()
        shutil.unpack_archive(str(ASSETS / archive), str(target))
model_dir = ASSETS / 'olmoearth-v1_1-tiny'
model_dir.mkdir(exist_ok=True)
for name, key in [('config.json','olmoearth_config_url'), ('weights.pth','olmoearth_weights_url')]:
    target = model_dir / name
    if not target.exists(): urlretrieve(config['assets'][key], target)
wheels = ASSETS / 'foundation-wheels'
wheels.mkdir(exist_ok=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'download', '-q', '--dest', str(wheels), 'universal-pathlib', 'omegaconf', 'fvcore', 'iopath', 'einops', 'huggingface_hub'])
RUNTIME_CONFIG = WORK / 'week7.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print({'project': str(PROJECT), 'work': str(WORK), 'assets': str(ASSETS)})


In [ ]:
install_offline_foundation_dependencies()
from spectrashift.train.week7 import freeze_week7_contracts, validate_foundation_assets

summary = freeze_week7_contracts(RUNTIME_CONFIG, ASSETS)
print(json.dumps(summary, indent=2))
assert summary['week7_contracts_complete']
assert summary['dinov2']['display_name'] == 'DINOv2 ViT-S/14'
assert summary['dinov2']['register_tokens'] == 0
assert summary['olmoearth']['feature_dimension'] == 192
assert summary['evaluation_labels_loaded'] is False
smoke = validate_foundation_assets(RUNTIME_CONFIG, WEEK5_CONTRACTS, WORK / 'week7_contracts_summary.json')
print(json.dumps(smoke, indent=2))
assert smoke['week7_adapter_smoke_complete'] and smoke['evaluation_labels_loaded'] is False
